# Exemplary script for network calculation of feed and reflux line network

- Creation of entire network from GIS data see file *networkModelling_example.py*

In [ ]:
### Imports
import os

import rasterio
import pandas as pd
from pathlib import Path
import geopandas as gp
import pandapipes as ppi
import numpy as np

from pandapipes.control import run_control
import geoppi

from geoppi import (transfer_LoadPoint_ppi, extract_FluidProperties_ppi, get_dict_from_aggregated_groups, implement_controllers, )

flp = Path(r"data/exampleNetwork")

In [65]:
### Set target feed temperature (°C)
tfeed_winter = 85
treflux_winter = 60
tfeed_summer = 70
treflux_summer = 55

### Load data and define parameters

net = ppi.from_pickle(flp / "exampleNetwork_ppi.p")
profile = pd.read_csv(flp / Path(r"heat_demand_profile.txt"), sep = "\t", header=0).to_numpy().flatten()

tfeed = np.ones(8760) * (tfeed_winter + 273.15)
tfeed[3500:5500] = (tfeed_summer + 273.15)

treflux = np.ones(8760) * (treflux_winter + 273.15)
treflux[3500:5500] = (treflux_summer + 273.15)
deltaT = tfeed - treflux

net.heat_consumer["demand_use_th"] = net.heat_consumer["demand_use_th"].fillna(15000)

In [66]:
net.heat_consumer["demand_use_th"]

0     25961.567566
1     19142.567642
2     15000.000000
3      7667.850586
4     15000.000000
5     14168.817352
6     22803.710403
7     25360.141846
8     15000.000000
9     35389.632507
10    19558.744629
11     8907.803123
12    17135.096741
13    10491.441170
14     3939.235977
15    12456.573486
16     2469.986469
17    45913.575439
18    68122.781494
19     2720.732306
20    86414.436646
21    15818.051788
22    15000.000000
23    27593.840637
24    26693.843811
25     5639.920830
26    22204.355942
27    19915.467839
28     4023.448608
29     8171.482910
30     5818.343628
31    14872.871704
32    15000.000000
33    15000.000000
34    15000.000000
35    15000.000000
36    15000.000000
37    15000.000000
Name: demand_use_th, dtype: float64

In [59]:
net.res_heat_consumer

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,reynolds,lambda,deltat_k,qext_w
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
np.nanmin(net.res_junction["p_bar"].values)

In [72]:
Qth_consumers

array([-122492.36145574, -124942.1644014 , -127391.96687987, ...,
             0.        ,       0.        ,       0.        ])

In [69]:
### Define containers for results
Qth_producers = np.zeros((2, 8760))
Qth_consumers = np.zeros(8760)

Pel_pump = np.zeros((2, 8760)) # El. effort for pumping

pmin = np.zeros(8760) # Min. pressure in network

tfeed_consumers = np.zeros((len(net.heat_consumer), 8760))

## Further parameters
cp_fluid, rho_fluid, nu_fluid, g = extract_FluidProperties_ppi(net = net, t = 0.5 * (tfeed_winter + treflux_winter))


### Start calculation
np.seterr(all='ignore')
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

for nn in range(100):
    # Set parameters
    net.heat_consumer["qext_w"] = net.heat_consumer["demand_use_th"] * profile[nn] * 1e03
    net.heat_consumer["controlled_mdot_kg_per_s"] = net.heat_consumer["qext_w"] / cp_fluid / deltaT[nn]

    net.controller.loc[0].object.target_T = treflux[nn]
    net.circ_pump_pressure["t_flow_k"] = tfeed[nn]

    run_control(net = net, max_iter = 25)

    # Write results
    tfeed_consumers[:,nn] = net.res_heat_consumer["t_from_k"].values
    pmin[nn] = np.nanmin(net.res_junction["p_bar"].values)

    Qth_producers[:, nn] = [
        (net.res_circ_pump_pressure["mdot_from_kg_per_s"].values * (net.res_circ_pump_pressure["t_to_k"] - net.res_circ_pump_pressure["t_from_k"]).values * cp_fluid)[0],
        (net.res_circ_pump_mass["mdot_from_kg_per_s"].values * (net.res_circ_pump_mass["t_to_k"] - net.res_circ_pump_mass["t_from_k"]).values * cp_fluid)[0]
    ]

    Qth_consumers[nn] = np.nansum(net.res_heat_consumer["mdot_from_kg_per_s"].values * (net.res_heat_consumer["t_to_k"] - net.res_heat_consumer["t_from_k"]).values * cp_fluid)






### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower than target temperature at heat_consumer idxs [30] ###

### Supply temperature is lower